In [2]:
# -*- coding: utf-8 -*-
"""10-k3s-architecture-diagrams-cytoscape.ipynb

Interactive architecture diagrams for your local k3s cluster using dash-cytoscape.
Now uses a standard Dash app with an iframe embed, eliminating JupyterDash issues.
"""

# %% [markdown]
# # 10 - k3s Architecture Diagrams with Cytoscape (Fixed)
# 
# This notebook builds on the networking deep dive (09) and the earlier matplotlib version (10) by creating **interactive** diagrams of your cluster’s components and their relationships using `dash-cytoscape`.
# 
# You can:
# - Pan, zoom, and drag nodes.
# - Click on nodes to inspect details (optional).
# - See the same generic k3s architecture, your actual cluster layout, and the application flow — now fully interactive.

# %% [markdown]
# ## Setup and Dependencies

# %%
import sys
import importlib.metadata as metadata

required = ["kubernetes", "pandas", "networkx", "dash", "dash-cytoscape"]
versions = {}
missing = []
for package in required:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = None
        missing.append(package)

print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
for package, version in versions.items():
    print(f"{package:20}: {version}")

if missing:
    print("Missing packages:", ", ".join(missing))
    print("Please install them with: pip install dash dash-cytoscape")
else:
    print("Dependency check passed.")

# %% [markdown]
# ## Kubernetes API Connection

# %%
from __future__ import annotations

import os
from pathlib import Path
import networkx as nx
import threading
import time

from kubernetes import client, config
from kubernetes.client import ApiException

# Reuse the config loading logic from notebook 09
def candidate_kubeconfigs() -> list[Path]:
    candidates: list[Path] = []
    env_value = os.getenv("KUBECONFIG")
    if env_value:
        for item in env_value.split(":"):
            path = Path(item).expanduser()
            if path.exists():
                candidates.append(path)
    for path in [Path("~/.kube/config").expanduser(), Path("/etc/rancher/k3s/k3s.yaml")]:
        if path.exists() and path not in candidates:
            candidates.append(path)
    return candidates

def load_k8s_config() -> tuple[str, str | None]:
    errors = []
    for kubeconfig_path in candidate_kubeconfigs():
        try:
            config.load_kube_config(config_file=str(kubeconfig_path))
            return "kubeconfig", str(kubeconfig_path)
        except Exception as exc:
            errors.append(f"{kubeconfig_path}: {exc}")
    try:
        config.load_incluster_config()
        return "in-cluster", None
    except Exception as exc:
        errors.append(f"in-cluster: {exc}")
    raise RuntimeError("Unable to load Kubernetes configuration. Tried: " + " | ".join(errors))

AUTH_MODE, KUBECONFIG_PATH = load_k8s_config()
print("Auth mode       :", AUTH_MODE)
print("Kubeconfig path :", KUBECONFIG_PATH)

# Initialize API clients
core = client.CoreV1Api()
apps = client.AppsV1Api()
networking = client.NetworkingV1Api()
version_api = client.VersionApi()

version_info = version_api.get_code()
print("Kubernetes ver. :", version_info.git_version)

# %% [markdown]
# ## Fetch Live Cluster Data

# %%
NAMESPACE = "llm-observability"

# Fetch all resources
nodes = core.list_node().items
all_pods = core.list_pod_for_all_namespaces().items
all_services = core.list_service_for_all_namespaces().items
all_deployments = apps.list_deployment_for_all_namespaces().items
all_statefulsets = apps.list_stateful_set_for_all_namespaces().items

# Filter for your application namespace
app_pods = [pod for pod in all_pods if pod.metadata.namespace == NAMESPACE]
app_services = [svc for svc in all_services if svc.metadata.namespace == NAMESPACE]
app_deployments = [dep for dep in all_deployments if dep.metadata.namespace == NAMESPACE]
app_statefulsets = [st for st in all_statefulsets if st.metadata.namespace == NAMESPACE]

print(f"Found {len(nodes)} node(s)")
print(f"Found {len(app_pods)} pods in namespace {NAMESPACE}")
print(f"Found {len(app_services)} services in namespace {NAMESPACE}")

# %% [markdown]
# ## Helper: Convert NetworkX Graph to Cytoscape Elements

# %%
def nx_to_cytoscape(G: nx.Graph, layout: str = "spring", **layout_kwargs) -> list:
    """
    Convert a NetworkX graph to Cytoscape elements.
    Returns a list of elements (nodes + edges).
    """
    # Compute positions
    if layout == "spring":
        pos = nx.spring_layout(G, **layout_kwargs)
    elif layout == "kamada_kawai":
        pos = nx.kamada_kawai_layout(G, **layout_kwargs)
    elif layout == "circular":
        pos = nx.circular_layout(G, **layout_kwargs)
    else:
        raise ValueError(f"Unsupported layout: {layout}")

    elements = []
    # Nodes
    for node, data in G.nodes(data=True):
        x, y = pos[node]
        # Scale to reasonable cytoscape coordinates (e.g., between -300 and 300)
        x = x * 500
        y = y * 500
        elements.append({
            'data': {
                'id': node,
                'label': data.get('label', node),
                'kind': data.get('kind', 'node'),
            },
            'position': {'x': x, 'y': y}
        })
    # Edges
    for u, v, data in G.edges(data=True):
        elements.append({
            'data': {
                'source': u,
                'target': v,
                'label': data.get('label', ''),
                'relation': data.get('relation', ''),
            }
        })
    return elements

# %% [markdown]
# ## Generic k3s Architecture Diagram (Interactive)

# %%
def build_generic_k3s_graph() -> nx.DiGraph:
    G = nx.DiGraph()
    # Add nodes
    G.add_node("k3s Server", kind="process", label="k3s Server")
    G.add_node("k3s Agent", kind="process", label="k3s Agent")
    G.add_node("API Server", kind="component")
    G.add_node("Scheduler", kind="component")
    G.add_node("Controller Manager", kind="component")
    G.add_node("Kine (etcd)", kind="component")
    G.add_node("kubelet", kind="component")
    G.add_node("containerd", kind="component")
    G.add_node("Flannel", kind="component")
    G.add_node("Tunnel Proxy", kind="component")
    for i in range(1, 4):
        G.add_node(f"Pod {i}", kind="pod", label=f"Pod {i}")

    # Add edges (relations)
    # Server -> components
    G.add_edge("k3s Server", "API Server", relation="contains")
    G.add_edge("k3s Server", "Scheduler", relation="contains")
    G.add_edge("k3s Server", "Controller Manager", relation="contains")
    G.add_edge("k3s Server", "Kine (etcd)", relation="contains")
    # Agent -> components
    G.add_edge("k3s Agent", "kubelet", relation="contains")
    G.add_edge("k3s Agent", "containerd", relation="contains")
    G.add_edge("k3s Agent", "Flannel", relation="contains")
    G.add_edge("k3s Agent", "Tunnel Proxy", relation="contains")
    # Dependencies between components
    G.add_edge("API Server", "Kine (etcd)", relation="stores data")
    G.add_edge("Scheduler", "API Server", relation="watches pods")
    G.add_edge("Controller Manager", "API Server", relation="watches resources")
    G.add_edge("kubelet", "API Server", relation="registers node")
    G.add_edge("kubelet", "containerd", relation="manages containers")
    G.add_edge("kubelet", "Flannel", relation="configures network")
    G.add_edge("Flannel", "Tunnel Proxy", relation="network plugin")
    # kubelet -> pods
    for i in range(1, 4):
        G.add_edge("kubelet", f"Pod {i}", relation="runs")
    return G

# %%
import dash
from dash import html
import dash_cytoscape as cyto
from IPython.display import IFrame, display

# Build the graph
G_generic = build_generic_k3s_graph()
elements_generic = nx_to_cytoscape(G_generic, layout="spring", seed=42, k=1.5)

# Define a stylesheet for node appearance based on "kind"
stylesheet = [
    {
        'selector': 'node',
        'style': {
            'label': 'data(label)',
            'text-valign': 'center',
            'text-halign': 'center',
            'background-color': '#2a9d8f',
            'border-width': 1,
            'border-color': '#264653',
            'width': '60px',
            'height': '60px',
        }
    },
    {
        'selector': 'node[kind = "process"]',
        'style': {
            'background-color': '#264653',
            'shape': 'rectangle',
            'width': '80px',
            'height': '40px',
        }
    },
    {
        'selector': 'node[kind = "pod"]',
        'style': {
            'background-color': '#e9c46a',
        }
    },
    {
        'selector': 'edge',
        'style': {
            'curve-style': 'bezier',
            'target-arrow-shape': 'triangle',
            'label': 'data(relation)',
            'font-size': '10px',
            'text-rotation': 'autorotate',
        }
    }
]

# Create a standard Dash app (not JupyterDash)
app_generic = dash.Dash(__name__)
app_generic.layout = html.Div([
    html.H3("Generic k3s Architecture", style={"textAlign": "center"}),
    cyto.Cytoscape(
        id='generic-k3s',
        elements=elements_generic,
        layout={'name': 'preset'},  # use positions from nx layout
        style={'width': '100%', 'height': '600px'},
        stylesheet=stylesheet,
    )
])

# Function to run the app in a background thread
def run_dash_app(app, port):
    app.run(debug=False, port=port, host='127.0.0.1', use_reloader=False)

# Start the server in a separate thread
thread_generic = threading.Thread(target=run_dash_app, args=(app_generic, 9050))
thread_generic.daemon = True
thread_generic.start()

# Give the server a moment to start
time.sleep(2)

# Display the app in an iframe
display(IFrame(src='http://localhost:9050', width='100%', height='650px'))

# %% [markdown]
# ## Concrete Cluster Diagram (Live Resources)

# %%
def build_cluster_graph() -> nx.DiGraph:
    G = nx.DiGraph()
    node_name = nodes[0].metadata.name
    G.add_node(f"Node: {node_name}", kind="node", label=node_name)

    # Control plane and agent (conceptual)
    G.add_node("k3s Server", kind="process", label="k3s Server")
    G.add_node("k3s Agent", kind="process", label="k3s Agent")
    G.add_edge(f"Node: {node_name}", "k3s Server", relation="runs")
    G.add_edge(f"Node: {node_name}", "k3s Agent", relation="runs")

    # Generic control plane components
    comps = ["API Server", "Scheduler", "Controller Manager", "Kine"]
    for comp in comps:
        G.add_node(comp, kind="component")
        G.add_edge("k3s Server", comp, relation="contains")

    # Generic node components
    node_comps = ["kubelet", "containerd", "Flannel", "Tunnel Proxy"]
    for comp in node_comps:
        G.add_node(comp, kind="component")
        G.add_edge("k3s Agent", comp, relation="contains")

    # Actual pods
    for pod in app_pods:
        pod_name = pod.metadata.name
        pod_node = f"Pod: {pod_name}"
        G.add_node(pod_node, kind="pod", label=pod_name)
        G.add_edge("kubelet", pod_node, relation="runs on")

    # Services and selectors
    for svc in app_services:
        svc_name = svc.metadata.name
        svc_node = f"Service: {svc_name}"
        G.add_node(svc_node, kind="service", label=svc_name)
        selector = svc.spec.selector or {}
        for pod in app_pods:
            pod_labels = pod.metadata.labels or {}
            if all(pod_labels.get(k) == v for k, v in selector.items()):
                G.add_edge(svc_node, f"Pod: {pod.metadata.name}", relation="selects")
        if svc.spec.type == "LoadBalancer" and svc.status.load_balancer.ingress:
            external_ip = svc.status.load_balancer.ingress[0].ip
            external_node = f"External: {external_ip}"
            G.add_node(external_node, kind="external", label=external_ip)
            G.add_edge(external_node, svc_node, relation="north-south")
    return G

# %%
# Build the graph for the cluster
G_cluster = build_cluster_graph()
elements_cluster = nx_to_cytoscape(G_cluster, layout="spring", seed=42, k=2.0)

# Extended stylesheet for external nodes
stylesheet_cluster = stylesheet + [
    {
        'selector': 'node[kind = "node"]',
        'style': {'background-color': '#264653', 'shape': 'rectangle', 'width': '100px', 'height': '50px'}
    },
    {
        'selector': 'node[kind = "external"]',
        'style': {'background-color': '#e76f51'}
    },
    {
        'selector': 'node[kind = "service"]',
        'style': {'background-color': '#f4a261'}
    }
]

# Create a new Dash app for the cluster diagram
app_cluster = dash.Dash(__name__)
app_cluster.layout = html.Div([
    html.H3("Live k3s Cluster Architecture", style={"textAlign": "center"}),
    cyto.Cytoscape(
        id='cluster-diagram',
        elements=elements_cluster,
        layout={'name': 'preset'},
        style={'width': '100%', 'height': '800px'},
        stylesheet=stylesheet_cluster,
    )
])

# Start server on a different port
thread_cluster = threading.Thread(target=run_dash_app, args=(app_cluster, 9051))
thread_cluster.daemon = True
thread_cluster.start()
time.sleep(2)
display(IFrame(src='http://localhost:9051', width='100%', height='850px'))

# %% [markdown]
# ## Application Flow Diagram

# %%
def build_app_flow_graph() -> nx.DiGraph:
    G = nx.DiGraph()
    services = ["open-webui", "langchain-demo", "ollama", "redis"]
    for svc in services:
        G.add_node(svc, kind="service", label=svc)
    G.add_edge("open-webui", "langchain-demo", relation="HTTP")
    G.add_edge("langchain-demo", "ollama", relation="HTTP")
    G.add_edge("open-webui", "redis", relation="TCP")
    G.add_node("Browser / Host", kind="external", label="Browser / Host")
    G.add_edge("Browser / Host", "open-webui", relation="LoadBalancer (8080)")
    return G

# %%
G_flow = build_app_flow_graph()
elements_flow = nx_to_cytoscape(G_flow, layout="spring", seed=42, k=1.0)

app_flow = dash.Dash(__name__)
app_flow.layout = html.Div([
    html.H3("Application Traffic Flow (llm-observability-stack)", style={"textAlign": "center"}),
    cyto.Cytoscape(
        id='flow-diagram',
        elements=elements_flow,
        layout={'name': 'preset'},
        style={'width': '100%', 'height': '500px'},
        stylesheet=stylesheet_cluster,  # reuse same styling
    )
])

thread_flow = threading.Thread(target=run_dash_app, args=(app_flow, 9052))
thread_flow.daemon = True
thread_flow.start()
time.sleep(2)
display(IFrame(src='http://localhost:9052', width='100%', height='550px'))

# %% [markdown]
# ## Optional: Interactive Callback Example
# 
# Uncomment the following code to add a callback that shows node details when clicked.

# %%
# # Example of adding an interactive callback (uncomment to enable)
# 
# app_interactive = dash.Dash(__name__)
# app_interactive.layout = html.Div([
#     html.H3("Click on any node to see its details"),
#     cyto.Cytoscape(
#         id='interactive-cyto',
#         elements=elements_cluster,
#         layout={'name': 'preset'},
#         style={'width': '100%', 'height': '700px'},
#         stylesheet=stylesheet_cluster,
#     ),
#     html.Div(id='node-details', style={'marginTop': '20px', 'padding': '10px', 'border': '1px solid #ccc'})
# ])
# 
# @app_interactive.callback(
#     dash.dependencies.Output('node-details', 'children'),
#     dash.dependencies.Input('interactive-cyto', 'tapNode')
# )
# def display_node_details(node):
#     if node:
#         node_data = node['data']
#         return f"Node ID: {node_data.get('id')}\nLabel: {node_data.get('label')}\nKind: {node_data.get('kind')}"
#     return "Click on a node to see its details."
# 
# thread_interactive = threading.Thread(target=run_dash_app, args=(app_interactive, 9053))
# thread_interactive.daemon = True
# thread_interactive.start()
# time.sleep(2)
# display(IFrame(src='http://localhost:9053', width='100%', height='750px'))

# %% [markdown]
# ## Conclusion
# 
# This notebook now uses a standard Dash app running in a background thread, embedded via an iframe. This avoids the `JupyterDash.run_server` issue and still provides fully interactive graphs.
# 
# You can explore the diagrams by panning, zooming, and clicking on nodes (if the callback is enabled). All data comes directly from your live k3s cluster.

Python executable: /usr/local/bin/python3.11
Python version   : 3.11.11
kubernetes          : 35.0.0
pandas              : 3.0.0
networkx            : 3.6.1
dash                : 4.0.0
dash-cytoscape      : 1.0.2
Dependency check passed.
Auth mode       : kubeconfig
Kubeconfig path : /etc/rancher/k3s/k3s.yaml
Kubernetes ver. : v1.34.5+k3s1
Found 1 node(s)
Found 5 pods in namespace llm-observability
Found 4 services in namespace llm-observability
